# Telemetry lab — look at native-rate rotor-speed telemetry yourself

Every rig with per-rotor RPS telemetry, at its **own** logging rate (never the 100 Hz campaign grid):
`neurobem` 400 Hz · `blackbird` 187 Hz · `vid` 1014 Hz · `nanobench` 100 Hz · `pitcn` 100 Hz ·
`dregon_measured` ~1001 Hz (45 Hz sample-and-hold) · `dregon_command` · `michaels` 29.4 Hz.

Logic lives in `telemetry_lab.py` (loader, airborne segmentation, high-pass, ACF, Welch, structure function).
Datasets are pulled by `dload` on first use; the rig-to-dataset map is `telemetry_lab.RIGS`.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import telemetry_lab as tl
plt.rcParams["figure.figsize"] = (12, 4)

## 1. Load a rig

In [ ]:
RIG = "blackbird"          # one of tl.RIGS
flights = tl.load(RIG, limit=None)
for f in flights:
    print(f"{f.name:40s} fs={f.fs:7.1f} Hz  {f.duration_s:7.1f} s  airborne={[(round(s.start/f.fs,1), round(s.stop/f.fs,1)) for s in f.airborne()]}")

## 2. Raw trajectories
Whole flight, all four rotors, plus a zoom you can move with `T0`/`DUR`.

In [ ]:
fl = flights[0]
T0, DUR = 20.0, 2.0          # zoom window, seconds

fig, ax = plt.subplots(2, 1, figsize=(12, 7))
for r in range(4):
    ax[0].plot(fl.t, fl.rps[r], lw=0.6, label=f"rotor {r}")
for s in fl.airborne():
    ax[0].axvspan(s.start / fl.fs, s.stop / fl.fs, color="k", alpha=0.05)
ax[0].set(title=f"{RIG} · {fl.name} · {fl.fs:.0f} Hz (grey = airborne)", ylabel="rev/s"); ax[0].legend(ncol=4)
m = (fl.t >= T0) & (fl.t < T0 + DUR)
for r in range(4):
    ax[1].step(fl.t[m], fl.rps[r, m], where="post", lw=0.8)
ax[1].set(xlabel="s", ylabel="rev/s", title=f"zoom {T0}–{T0+DUR} s (step plot shows the logger's hold pattern)")
plt.tight_layout()

## 3. What a 31.25 Hz label cannot carry
`nu_res` = speed error above 15.6 Hz (zero-phase Butterworth), in rad/s. Alternative: `tl.label_residual` = native minus resample-to-31.25-Hz-and-back.

In [ ]:
seg = fl.airborne()[0]
rps_seg = fl.rps[:, seg]
nu = 2 * np.pi * rps_seg                       # rad/s
nu_res = tl.highpass(nu, fl.fs)                # above the label band
nu_lab = tl.label_residual(nu, fl.fs)          # the exact label residual (linear resampler)
print("residual rms [rad/s]  high-pass:", nu_res.std(axis=1).round(3), "  label residual:", nu_lab.std(axis=1).round(3))

t = np.arange(rps_seg.shape[1]) / fl.fs
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(t, rps_seg[0], lw=0.6); ax[0].set(ylabel="rev/s", title="rotor 0, airborne segment")
ax[1].plot(t, nu_res[0] / (2 * np.pi), lw=0.6, label="high-pass 15.6 Hz")
ax[1].plot(t, nu_lab[0] / (2 * np.pi), lw=0.6, alpha=0.6, label="label residual")
ax[1].set(xlabel="s", ylabel="rev/s", xlim=(T0 - seg.start / fl.fs, T0 - seg.start / fl.fs + DUR)); ax[1].legend()
plt.tight_layout()

## 4. Is the residual white or correlated?
Autocorrelation of the residual speed error against the **filtered-white null** (white noise through the same high-pass — the shape the filter alone produces). Band = ±2/√N.

In [ ]:
lag, rho = tl.acf(nu_res[0], fl.fs)
_, rho_null = tl.acf(tl.white_null(nu_res.shape[1], fl.fs), fl.fs)
band = 2 / np.sqrt(nu_res.shape[1])
plt.figure(figsize=(12, 4))
plt.plot(lag * 1e3, rho, label="measured residual (rotor 0)")
plt.plot(lag * 1e3, rho_null, "--", label="white noise through the same filter")
plt.axhspan(-band, band, color="k", alpha=0.08)
plt.xlim(0, 200); plt.xlabel("lag [ms]"); plt.ylabel("ρ"); plt.legend(); plt.title("ACF of the >15.6 Hz speed residual")
for ms in (10, 25, 50, 100):
    print(f"rho({ms} ms) = {np.interp(ms/1e3, lag, rho):+.3f}   null {np.interp(ms/1e3, lag, rho_null):+.3f}")

## 5. Speed-error PSD
Log-log slope: 0 = white, −2 = OU tail. The vertical line is the label band edge.

In [ ]:
f, P = tl.welch(nu[0] - nu[0].mean(), fl.fs)
lo, hi = tl.loglog_slope(f, P, (2, 15)), tl.loglog_slope(f, P, (16, 0.45 * fl.fs))
plt.figure(figsize=(8, 5))
plt.loglog(f[1:], P[1:], lw=0.8)
plt.axvline(15.6, color="r", ls="--", label="label band edge 15.6 Hz")
plt.xlabel("Hz"); plt.ylabel("rad²/s² / Hz"); plt.legend()
plt.title(f"slope 2–15 Hz = {lo:+.2f}   slope 16–{0.45*fl.fs:.0f} Hz = {hi:+.2f}")

## 6. Phase error implied by the residual
θ_res = ∫ν_res dt. Structure function `Var[θ(t+τ) − θ(t)]`: slope 2 = smooth wobble, slope 1 = diffusion (Wiener), flat = bounded.

In [ ]:
theta = tl.phase(nu_res[0], fl.fs)
lags = np.geomspace(1 / fl.fs, 2.0, 40)
S = tl.structure_function(theta, fl.fs, lags)
print(f"rms theta_res = {theta.std():.4f} rad at k=1  ->  {10*theta.std():.3f} rad at k=10, {30*theta.std():.3f} rad at k=30")
plt.figure(figsize=(8, 5))
plt.loglog(lags, S, "o-", ms=3, label="measured")
for slope, c in ((2, "g"), (1, "r")):
    plt.loglog(lags, S[3] * (lags / lags[3]) ** slope, "--", c=c, lw=0.8, label=f"slope {slope}")
plt.xlabel("τ [s]"); plt.ylabel("Var[Δθ] [rad²]"); plt.legend(); plt.title("structure function of the residual phase")

## 7. All rigs at once (one number each)
Pooled over rotors and airborne segments; takes ~1–2 min for the big rigs.

In [ ]:
rows = []
for rig in ["neurobem", "blackbird", "vid", "nanobench", "pitcn", "dregon_measured", "dregon_command"]:
    fls = tl.load(rig, limit=3)
    rhos, slopes, rms = [], [], []
    for fl_ in fls:
        for s in fl_.airborne():
            nu_ = 2 * np.pi * fl_.rps[:, s]
            res = tl.highpass(nu_, fl_.fs)
            for r in range(4):
                lag_, rho_ = tl.acf(res[r], fl_.fs, 0.2)
                rhos.append(np.interp(0.05, lag_, rho_))
                f_, P_ = tl.welch(nu_[r] - nu_[r].mean(), fl_.fs)
                slopes.append(tl.loglog_slope(f_, P_, (16, 0.45 * fl_.fs)))
                rms.append(res[r].std())
    rows.append((rig, fls[0].fs, np.mean(rms), np.mean(rhos), np.nanmean(slopes)))
print(f"{'rig':16s}{'fs':>8s}{'res rms rad/s':>15s}{'rho(50ms)':>11s}{'slope>16Hz':>12s}")
for r in rows:
    print(f"{r[0]:16s}{r[1]:8.0f}{r[2]:15.3f}{r[3]:11.3f}{r[4]:12.2f}")